# 03. Train the normal-only skeleton JEPA

We will follow one prediction through the context encoder, teacher, and loss, then train a protected normal-only reference and inspect collapse diagnostics.

## Run this notebook independently

From `experiments/aaai`, run:

```bash
uv sync
uv run jupyter lab
```

Select the kernel from this environment. Python 3.12 and all notebook libraries are declared in `pyproject.toml`. Every notebook repeats its setup and loads its own inputs. No variable from another notebook is required.

A fresh installation defaults to `AAAI_DEMO=1` for a small synthetic software check. This prepared workspace uses `AAAI_DEMO=0` after real-data ingestion and validation. Its condition parameters are invented and its scores are not research evidence. Set `AAAI_DEMO=0` in the repository root `.env` after the ingestion workflow has produced a complete pose corpus. All notebooks use the same root `.env`. No API key is needed for JEPA or Random Forest training. Restart the kernel after changing configuration.

Only the root environment file contains credentials. `load_env` and `find_env` below are aliases for the real `python-dotenv` functions, `load_dotenv` and `find_dotenv`.

## Historical tutorial and current study

This notebook preserves the earlier tutorial's code, observations and experimental scope. Its original result identities remain historical; they are not measurements of the new retained-predictor or U/J/A study. Start the current independent tutorial sequence at [Notebook 12](12_observation_population_and_video_context.ipynb), then continue through [Notebook 17](17_failure_analysis_and_manuscript_decision.ipynb). The [current protocol](docs/iteration02/protocol.md) and [current results](docs/iteration02/results.md) identify the verified new run. The [draft01 manuscript](docs/draft01/manuscript.md) remains the record of the completed earlier study.

The current primary question is normal versus the four requested condition families, with all other folders retained in a mandatory broad secondary analysis. It tests opposite-leg information beyond own-leg geometry and extra capacity. Uniform masking remains the reference, while J and A are declared motion-independent alternatives with matched budgets. Read the newer notebooks for their support rules and fixed evaluation banks. Do not combine old and new scores or denominators.

In [ ]:
from pathlib import Path
import os, sys
# Find the repository from either its root or this notebook directory.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'ambient').is_dir() and (p / 'data/gavd').is_dir())
BASE = ROOT / 'experiments/aaai'
os.chdir(ROOT)
from dotenv import load_dotenv as load_env, find_dotenv as find_env
load_env(find_env())  # requested spelling, using aliases for python-dotenv's real API
os.chdir(BASE)
if str(BASE) not in sys.path: sys.path.insert(0, str(BASE))
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from aaai_gait.settings import Settings
from aaai_gait.data import build_manifest, grouped_split, LABELS
from aaai_gait.media import show_video_context, plot_motion
settings = Settings()
torch.set_num_threads(2)
manifest = grouped_split(build_manifest(ROOT), seed=settings.seed)
print('Mode:', 'SYNTHETIC SOFTWARE DEMONSTRATION' if settings.demo else 'GAVD RESEARCH RUN')
print('Current annotation sequences:', len(manifest), 'unique sources:', manifest.source_id.nunique())

## Connect the experiment to walking

Watch a short annotated normal training sequence when local media is available. The preview is an observed video. Synthetic arrays used below are a separate software demonstration.

In [ ]:
video_row = show_video_context(manifest, frame_number_base=settings.frame_number_base)

In [ ]:
from aaai_gait.workflow import prepare_corpus
x, meta, provenance = prepare_corpus(settings)
print('Window tensor:', x.shape, '[windows, frames, selected joints, XY]')
display(meta.groupby(['split','label']).agg(windows=('sequence_id','size'), sequences=('sequence_id','nunique'), sources=('source_id','nunique')))

## 1. Separate context, target, and predictor

The online encoder receives visible tokens only. Attention therefore cannot read hidden coordinates. Invalid patches are also excluded as attention keys, and the loss uses valid hidden targets only. The predictor restores the time-joint grid using learned queries and predicts a distribution over feature channels. The full-input teacher supplies the target distribution without receiving gradients.

![Branches](images/jepa_branches.svg)

For token `i`, let `h_i` be the teacher feature and `p_i` the predictor output. The target is `softmax((h_i - center) / target_temperature)`. The prediction is `softmax(p_i / student_temperature)`. Cross-entropy is averaged only over hidden positions. The moving center discourages a persistent dominant teacher channel.

In [ ]:
from aaai_gait.model import ModelConfig, SJEPA
from aaai_gait.training import seed_everything
seed_everything(settings.seed)
example=SJEPA(ModelConfig())
normal_train=(meta.split.eq('train')&meta.label.eq('normal')).to_numpy()
example_batch=torch.as_tensor(x[normal_train][:4])
losses=example.loss(example_batch,generator=torch.Generator().manual_seed(3))
print({k:round(float(losses[k].detach()),4) for k in ['loss','prediction','vicreg','projection_std','embedding_std']})
losses['loss'].backward()
assert all(p.grad is None for p in example.target_encoder.parameters())
print('Teacher gradients are absent.')

## 2. Add anti-collapse regularization

Two randomly masked views of the same walking window produce pooled online features. An unnormalized projector feeds VICReg. Its invariance term encourages view agreement. Its variance term penalizes dimensions with low batch variation. Its covariance term reduces redundant dimensions.

The loss is `prediction + 0.1 * VICReg`, with internal invariance and variance weights of 25 and covariance weight of 1. These are starting settings. VICReg is not a supervised repulsion between clinical classes. A small prediction loss alone does not prove useful learning.

![VICReg](images/vicreg.svg)

## 3. Run normal-only pretraining

The training function selects only rows with `split == train` and `label == normal`. Source-balanced sampling prevents a long video from dominating. Each optimizer step updates the online network first, then performs `teacher = 0.996 * teacher + 0.004 * online`. Target-center statistics come from training batches only.

The helper stores configuration, source split identity, and data identity. It refuses to overwrite an existing anchor. Reopening the notebook reuses a compatible checkpoint. A new corpus or implementation produces a new artifact directory.

In [ ]:
from aaai_gait.workflow import normal_anchor
anchor,history=normal_anchor(x,meta,provenance,settings)
display(pd.DataFrame(history))
print('Training sources used:',meta.loc[normal_train,'source_id'].nunique())
print('Artifacts:',provenance['run_dir'])

## 4. Check the actual encoder

Inspect both the projector's standard deviation and the encoder's variation. A projector can hide problems in the representation it receives. Effective rank describes how many directions carry variation, but it is bounded by the number of examples minus one. Neither rank nor a visually separated plot establishes clinical validity.

In [ ]:
from aaai_gait.training import embed_windows, representation_diagnostics
train=meta.split.eq('train').to_numpy()
z=embed_windows(anchor,x[train],batch_size=settings.batch_size)
display(representation_diagnostics(z))
plt.figure(figsize=(7,3));plt.plot([row['prediction'] for row in history],label='Latent cross-entropy');plt.plot([row['vicreg'] for row in history],label='VICReg');plt.xlabel('Epoch');plt.legend();plt.title('Synthetic check' if settings.demo else 'Normal training losses');plt.show()

## 5. Know what this model predicts

The predictor outputs latent features. It does not yet generate joint coordinates or a video of a future gait. Normal-only masked completion also differs from a dedicated past-to-future forecasting model. Notebook 05 makes that distinction executable. The frozen online encoder supplies features for the condition probes in later notebooks.

## What to record before a paper run

Save the current annotation hash, source assignments, retained windows, exclusions, software versions, model configuration, seed, and exact command. Keep synthetic artifacts separate. Revisit the video when an apparent anomaly could instead be a crop error, occlusion, or viewpoint change.

The [research review](docs/literature_review.md) explains source methods and adaptations. The [paper plan](docs/paper_plan.md) defines claims, ablations, and limits.